In [1]:
import os
import re
import math
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter, defaultdict

In [2]:
INPUT_DIR = Path("/kaggle/input")

def find_file(filename):
    matches = list(INPUT_DIR.rglob(filename))
    if len(matches) == 0:
        raise FileNotFoundError(f"{filename} not found. Make sure competition data is added.")
    return matches[0]

train_path = find_file("train.csv")
test_path = find_file("test.csv")
sample_path = find_file("sample_submission.csv")
laws_path = find_file("laws_de.csv")

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample = pd.read_csv(sample_path)
laws = pd.read_csv(laws_path)

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Sample shape:", sample.shape)
print("Laws shape:", laws.shape)

print("Train columns:", train.columns.tolist())
print("Test columns:", test.columns.tolist())
print("Sample columns:", sample.columns.tolist())
print("Laws columns:", laws.columns.tolist())

Train shape: (1139, 3)
Test shape: (40, 2)
Sample shape: (2, 2)
Laws shape: (175933, 3)
Train columns: ['query_id', 'query', 'gold_citations']
Test columns: ['query_id', 'query']
Sample columns: ['query_id', 'predicted_citations']
Laws columns: ['citation', 'text', 'title']


In [3]:
question_col = "query"
gold_col = "gold_citations"
citation_col = "citation"
text_col = "text"
title_col = "title"

In [4]:
laws = laws.dropna(subset=[citation_col, text_col]).reset_index(drop=True)

law_citations = laws[citation_col].astype(str).str.strip().tolist()

law_texts = (
    laws[citation_col].fillna("").astype(str) + " " +
    laws[title_col].fillna("").astype(str) + " " +
    laws[text_col].fillna("").astype(str)
).tolist()

print("Number of law documents:", len(law_texts))
print("Example citation:", law_citations[0])
print("Example law text:", law_texts[0][:300])

Number of law documents: 175933
Example citation: Art. 1 112
Example law text: Art. 1 112 Übereinkunft vom 22. Juni 1875 zwischen dem Schweizerischen Bundesrate und dem Einwohnergemeinderate der Stadt Bern betreffend die Leistungen der Stadt Bern an den Bundessitz Die Einwohnergemeinde Bern tritt der Schweizerischen Eidgenossenschaft unentgeltlich als Eigentum ab:a. Das Gebäud


In [5]:
def tokenize(text):
    text = str(text).lower()
    tokens = re.findall(r"[a-zA-ZÀ-ÿ0-9]+", text)
    return tokens

In [6]:
class BM25:
    def __init__(self, documents, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        
        self.docs = [tokenize(doc) for doc in documents]
        self.N = len(self.docs)
        self.doc_len = np.array([len(doc) for doc in self.docs])
        self.avgdl = self.doc_len.mean()
        
        self.df = Counter()
        self.postings = defaultdict(list)
        
        for i, doc in enumerate(self.docs):
            tf = Counter(doc)
            for term, freq in tf.items():
                self.df[term] += 1
                self.postings[term].append((i, freq))
        
        self.idf = {}
        for term, df in self.df.items():
            self.idf[term] = math.log(1 + (self.N - df + 0.5) / (df + 0.5))
    
    def get_scores(self, query):
        query_terms = tokenize(query)
        scores = np.zeros(self.N)
        
        for term in query_terms:
            if term not in self.postings:
                continue
            
            idf = self.idf.get(term, 0)
            
            for doc_id, freq in self.postings[term]:
                dl = self.doc_len[doc_id]
                numerator = freq * (self.k1 + 1)
                denominator = freq + self.k1 * (1 - self.b + self.b * dl / self.avgdl)
                scores[doc_id] += idf * numerator / denominator
        
        return scores

In [7]:
bm25 = BM25(law_texts)

print("BM25 index built successfully.")

BM25 index built successfully.


In [8]:
legal_dictionary = {
    # Contract / obligation law
    "contract": "Vertrag Vereinbarung Obligation OR",
    "agreement": "Vertrag Vereinbarung OR",
    "obligation": "Obligation Schuld OR",
    "liability": "Haftung Verantwortlichkeit Schaden OR",
    "damage": "Schaden Ersatz Haftung OR",
    "damages": "Schadenersatz Haftung OR",
    "compensation": "Entschädigung Schadenersatz OR",
    "termination": "Kündigung Beendigung Auflösung OR",
    "unfair termination": "missbräuchliche Kündigung OR",
    "employment": "Arbeitsvertrag Arbeitnehmer Arbeitgeber OR",
    "employee": "Arbeitnehmer OR",
    "employer": "Arbeitgeber OR",
    "lease": "Miete Mietvertrag OR",
    "rent": "Miete Mietzins OR",
    "sale": "Kauf Kaufvertrag OR",

    # Civil code / family / inheritance
    "marriage": "Ehe Ehegatte ZGB",
    "divorce": "Scheidung Ehe ZGB",
    "child": "Kind Eltern ZGB",
    "parent": "Eltern Kind ZGB",
    "inheritance": "Erbschaft Erbe Erbrecht ZGB",
    "estate": "Nachlass Erbschaft ZGB",
    "property": "Eigentum Grundstück Besitz ZGB",
    "ownership": "Eigentum Besitz ZGB",
    "land register": "Grundbuch ZGB",
    "personality": "Persönlichkeit Persönlichkeitsschutz ZGB",

    # Criminal law
    "criminal": "strafbar Strafe Strafgesetzbuch StGB",
    "crime": "Straftat strafbar StGB",
    "offence": "Straftat strafbar StGB",
    "murder": "Tötung Mord StGB",
    "theft": "Diebstahl StGB",
    "fraud": "Betrug StGB",
    "assault": "Körperverletzung StGB",

    # Federal court / procedure
    "appeal": "Beschwerde Berufung BGG",
    "federal court": "Bundesgericht BGG",
    "court": "Gericht Bundesgericht BGG",
    "decision": "Entscheid Urteil BGG",
    "judgment": "Urteil Entscheid BGG",
    "admissible": "zulässig Zulässigkeit BGG",

    # Constitution
    "constitutional": "Verfassung Bundesverfassung BV",
    "municipality": "Gemeinde BV",
    "canton": "Kanton BV",
    "fundamental right": "Grundrecht BV",

    # Environment
    "environment": "Umwelt Umweltschutz USG",
    "environmental": "Umwelt Umweltschutz USG",
    "impact assessment": "Umweltverträglichkeitsprüfung UVPV",
    "pollution": "Verschmutzung Emission Immission USG",

    # Insurance / accident
    "insurance": "Versicherung UVG",
    "accident": "Unfall Unfallversicherung UVG",

    # International private law / arbitration
    "international": "international Ausland IPRG",
    "foreign": "Ausland international IPRG",
    "arbitration": "Schiedsgericht Schiedsverfahren IPRG",
    "arbitral": "Schiedsgericht Schiedsverfahren IPRG",
}

In [9]:
law_area_keywords = {
    "OR": [
        "contract", "agreement", "obligation", "liability", "damage", "damages",
        "compensation", "termination", "employment", "employee", "employer",
        "lease", "rent", "sale"
    ],
    "ZGB": [
        "marriage", "divorce", "child", "parent", "inheritance", "estate",
        "property", "ownership", "land register", "personality"
    ],
    "StGB": [
        "criminal", "crime", "offence", "murder", "theft", "fraud", "assault"
    ],
    "BGG": [
        "appeal", "federal court", "court", "decision", "judgment", "admissible"
    ],
    "BV": [
        "constitutional", "municipality", "canton", "fundamental right"
    ],
    "USG": [
        "environment", "environmental", "pollution"
    ],
    "UVPV": [
        "impact assessment", "environmental impact"
    ],
    "UVG": [
        "insurance", "accident"
    ],
    "IPRG": [
        "international", "foreign", "arbitration", "arbitral"
    ],
}


def detect_target_laws(query):
    query_lower = str(query).lower()
    target_laws = []

    for law_code, keywords in law_area_keywords.items():
        for keyword in keywords:
            if keyword in query_lower:
                target_laws.append(law_code)
                break

    return list(set(target_laws))


def expand_query(query):
    query_lower = str(query).lower()
    extra_terms = []

    for eng, ger in legal_dictionary.items():
        if eng in query_lower:
            extra_terms.append(ger)

    target_laws = detect_target_laws(query)
    extra_terms.extend(target_laws)

    expanded = str(query) + " " + " ".join(extra_terms)
    return expanded, target_laws

In [10]:
def extract_citations_from_query(query):
    query = str(query)

    law_codes = r"(ZGB|OR|StGB|BGG|BV|USG|UVPV|UVG|IPRG)"
    
    pattern = rf"Art\.?\s*\d+[a-zA-Z]?(?:\s*Abs\.?\s*\d+)?\s*{law_codes}"
    
    matches = re.findall(pattern, query)
    
    # re.findall with group only returns law code, so use finditer instead
    citations = []
    for m in re.finditer(pattern, query):
        citation = m.group(0)
        citation = citation.replace("Art ", "Art. ")
        citation = re.sub(r"\s+", " ", citation).strip()
        citations.append(citation)

    valid_law_citations = set(law_citations)
    citations = [c for c in citations if c in valid_law_citations]

    return citations

In [11]:
def parse_citations(x):
    if pd.isna(x):
        return set()
    return set(c.strip() for c in str(x).split(";") if c.strip())


def f1_score_citations(gold, pred):
    gold = set(gold)
    pred = set(pred)
    
    if len(gold) == 0 and len(pred) == 0:
        return 1.0
    
    if len(gold) == 0 or len(pred) == 0:
        return 0.0
    
    tp = len(gold & pred)
    precision = tp / len(pred)
    recall = tp / len(gold)
    
    if precision + recall == 0:
        return 0.0
    
    return 2 * precision * recall / (precision + recall)

In [12]:
def retrieve_ranked_citations(query, top_k_docs=200, max_citations=30):
    
    # 1. Extract citations directly from query, if any
    direct_citations = extract_citations_from_query(query)

    # 2. Expand English query with German legal terms
    expanded_query, target_laws = expand_query(query)

    # 3. BM25 scoring
    scores = bm25.get_scores(expanded_query)

    # 4. Boost documents from likely legal codes
    if target_laws:
        for i, citation in enumerate(law_citations):
            for law_code in target_laws:
                if citation.endswith(" " + law_code) or (" " + law_code) in citation:
                    scores[i] *= 1.35
                    scores[i] += 0.5

    # 5. Get top documents
    top_k_docs = min(top_k_docs, len(scores))
    top_idx = np.argpartition(scores, -top_k_docs)[-top_k_docs:]
    top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]

    selected = []
    seen = set()

    # 6. Add direct citations first
    for citation in direct_citations:
        if citation not in seen:
            selected.append(citation)
            seen.add(citation)

    # 7. Add BM25 citations
    for idx in top_idx:
        citation = law_citations[idx]

        if citation not in seen:
            selected.append(citation)
            seen.add(citation)

        if len(selected) >= max_citations:
            break

    return selected

In [13]:
max_cit_values = [3, 5, 8, 10, 15, 20, 30]

valid = train.sample(min(50, len(train)), random_state=42).copy()
max_needed = max(max_cit_values)

all_predictions = []

for _, row in valid.iterrows():
    query = row[question_col]
    pred = retrieve_ranked_citations(
        query,
        top_k_docs=200,
        max_citations=max_needed
    )
    all_predictions.append(pred)

best_f1 = -1
best_max_citations = None

for max_cit in max_cit_values:
    scores_list = []
    
    for i, (_, row) in enumerate(valid.iterrows()):
        gold = parse_citations(row[gold_col])
        pred = all_predictions[i][:max_cit]
        score = f1_score_citations(gold, pred)
        scores_list.append(score)
    
    avg_f1 = np.mean(scores_list)
    print("max_citations =", max_cit, "Average F1 =", avg_f1)
    
    if avg_f1 > best_f1:
        best_f1 = avg_f1
        best_max_citations = max_cit

print("\nBest max_citations:", best_max_citations)
print("Best validation F1:", best_f1)

max_citations = 3 Average F1 = 0.04933333333333333
max_citations = 5 Average F1 = 0.042678571428571434
max_citations = 8 Average F1 = 0.02958730158730159
max_citations = 10 Average F1 = 0.02828250710603652
max_citations = 15 Average F1 = 0.023881207028265853
max_citations = 20 Average F1 = 0.018631180009903416
max_citations = 30 Average F1 = 0.012988416624240053

Best max_citations: 3
Best validation F1: 0.04933333333333333


In [14]:
test_predictions = []

for _, row in test.iterrows():
    query = row[question_col]
    pred = retrieve_ranked_citations(
        query,
        top_k_docs=200,
        max_citations=best_max_citations
    )
    test_predictions.append(pred)

submission = pd.DataFrame({
    "query_id": test["query_id"],
    "predicted_citations": [";".join(cites) for cites in test_predictions]
})

display(submission.head())
print(submission.shape)
print(submission.columns.tolist())

submission.to_csv("submission.csv", index=False)

print("Improved submission.csv created.")

,query_id,predicted_citations
0,test_001,Art. 197e Abs. 2 AVO;Art. 3 Abs. 1 221.434;Art...
1,test_002,Art. 3 Abs. 1 221.434;Art. 197e Abs. 2 AVO;Art...
2,test_003,Art. 3 Abs. 1 221.434;Art. 197e Abs. 2 AVO;Art...
3,test_004,Art. 3 Abs. 1 221.434;Art. 197e Abs. 2 AVO;Art...
4,test_005,Art. 3 Abs. 1 221.434;Art. 197e Abs. 2 AVO;Art...


(40, 2)
['query_id', 'predicted_citations']
Improved submission.csv created.
